# Session 1 — TPU Benchmark: Batch Size Scaling

## Background

The GPU executes eagerly on thousands of SIMT cores; each step dispatches work immediately. The TPU's systolic array (MXU) processes large matrix blocks in a fixed pipeline — it is most efficient when there is enough work to fill every cell. Batch size is the simplest lever that controls how much work arrives per step and therefore which execution model wins. Session 1 asks: where do these two devices diverge, and by how much?

## Goal

At the end of this session, participants will have run the same training loop on CPU, GPU, and TPU without changing model code. They will be able to read a throughput curve and explain why the GPU saturates early while the TPU scales near-linearly — grounding the interpretation in memory bandwidth and MXU utilization, not hardware marketing.

## Question

At what batch size does the TPU overtake the GPU, and how large does the gap become?

---

**Runtime:** TPU — `Runtime → Change runtime type → TPU`

After running, results are saved to `results/tpu.json` and loaded automatically by `04-analysis.ipynb`.

**Note:** The first few steps are slow — XLA is compiling the graph. Step time stabilizes after warmup.

In [1]:
# pip install torch

In [2]:
import subprocess, sys, time
import torch, torch.nn as nn

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch==2.9.0", "torch_xla[tpu]==2.9.0",
        "-f", "https://storage.googleapis.com/libtpu-releases/index.html"
    ])
    import torch_xla
    import torch_xla.core.xla_model as xm
    import torch_xla.runtime as xr

# ── TPU info ──────────────────────────────────────────────────────────────────
device = torch_xla.device()

try:
    from torch_xla._internal import tpu as _tpu
    tpu_env  = _tpu.get_tpu_env()
    chip     = tpu_env.get("ACCELERATOR_TYPE") or tpu_env.get("TPU_ACCELERATOR_TYPE", "unknown")
    n_chips  = xr.global_runtime_device_count()
except Exception:
    chip, n_chips = "unknown", "unknown"

print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"torch_xla : {torch_xla.__version__}")
print(f"Device    : {device}")
print(f"TPU chip  : {chip}")
print(f"N chips   : {n_chips}")

Python    : 3.12.12
PyTorch   : 2.9.0+cu128
torch_xla : 2.9.0
Device    : xla:0
TPU chip  : v5litepod-1
N chips   : 1


In [3]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
D_MODEL, N_HEAD, DIM_FF, SEQ_LEN = 512, 8, 2048, 128
N_STEPS, WARMUP = 50, 5

# ── Model ─────────────────────────────────────────────────────────────────────
class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn  = nn.MultiheadAttention(D_MODEL, N_HEAD, batch_first=True)
        self.ff    = nn.Sequential(nn.Linear(D_MODEL, DIM_FF), nn.GELU(), nn.Linear(DIM_FF, D_MODEL))
        self.norm1 = nn.LayerNorm(D_MODEL)
        self.norm2 = nn.LayerNorm(D_MODEL)
    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x)[0])
        x = self.norm2(x + self.ff(x))
        return x

# ── Benchmark function ────────────────────────────────────────────────────────
def benchmark(batch_size):
    model     = TransformerBlock().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        x = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)
        y = torch.randn(batch_size, SEQ_LEN, D_MODEL, device=device)
        torch_xla.sync()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        torch_xla.sync()
        if step >= WARMUP:
            elapsed += time.perf_counter() - t0
    return (N_STEPS * batch_size) / elapsed

print("Model and benchmark function defined.")

Model and benchmark function defined.


In [4]:
BATCH_SIZES = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
results     = {}

for bs in BATCH_SIZES:
    try:
        print(f"  [TPU] batch={bs:>4} ... ", end="", flush=True)
        tput = benchmark(bs)
        results[bs] = round(tput, 1)
        print(f"{tput:,.0f} samples/sec")
    except RuntimeError as e:
        if "out of memory" in str(e).lower() or "resource" in str(e).lower():
            print("OOM — stopping.")
            break
        raise

print("\nDone.")
print("\n── Copy this into 04-analysis.ipynb ──────────────────")
print(f"TPU_RESULTS = {results}")

  [TPU] batch=   4 ... 371 samples/sec
  [TPU] batch=   8 ... 753 samples/sec
  [TPU] batch=  16 ... 1,498 samples/sec
  [TPU] batch=  32 ... 2,975 samples/sec
  [TPU] batch=  64 ... 5,946 samples/sec
  [TPU] batch= 128 ... 11,877 samples/sec
  [TPU] batch= 256 ... 23,853 samples/sec
  [TPU] batch= 512 ... 47,658 samples/sec
  [TPU] batch=1024 ... 95,703 samples/sec

Done.

── Copy this into 04-analysis.ipynb ──────────────────
TPU_RESULTS = {4: 371.3, 8: 753.0, 16: 1497.5, 32: 2975.1, 64: 5945.9, 128: 11876.9, 256: 23852.9, 512: 47658.1, 1024: 95703.1}


In [ ]:
import json, os
from datetime import datetime, timezone

def save_results(hardware, results, device_name="", results_dir="results"):
    os.makedirs(results_dir, exist_ok=True)
    payload = {
        "hardware":    hardware,
        "device_name": device_name,
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "results":     {str(k): v for k, v in sorted(results.items())},
    }
    path = os.path.join(results_dir, f"{hardware.lower()}.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path

path = save_results("TPU", results, device_name=chip, results_dir="results")
print(f"Saved → {path}")